# L04 · 원 논문 GKD 해부

## Goal

**예상 시간:** 45분 · **경로:** fast, full

- GKD의 lambda와 beta를 분리한다
- 일반화 JSD를 구현한다
- gradient 경로를 추적한다

### 현재 위치: L03 → **L04** → L05

```text
Prompt/Data -> state source -> ... -> L04 -> ... -> fair evaluation
```

Alt text: The course map highlights L04 between its prerequisite and next lesson; every method remains connected to the same evaluation stage.

## Setup

In [1]:
LESSON_ID = "L04"
from pathlib import Path
import sys
import torch

repo_root = Path.cwd()
if not (repo_root / "src").exists():
    repo_root = Path.cwd().parents[1]
sys.path.insert(0, str(repo_root / "src"))

import opd_study
from opd_study.device import resolve_device
from opd_study.utils import seed_everything

seed_everything(42)
device_report = resolve_device("cpu")
print({"lesson": LESSON_ID, "opd_study": opd_study.__version__,
       "torch": torch.__version__, "device": device_report.selected,
       "profile": "toy", "network": "not required"})

{'lesson': 'L04', 'opd_study': '0.1.0.dev0', 'torch': '2.13.0', 'device': 'cpu', 'profile': 'toy', 'network': 'not required'}


## Steps

### 1/3 · 8–12 min

원 GKD에는 두 축이 있다. lambda는 student가 만든 state의 비율이고 beta는 그 state에서 비교할 divergence다. 둘을 한 hyperparameter처럼 설명하면 안 된다.

그림 대체 설명: 출력의 label과 숫자는 색 없이도 읽을 수 있다.

### 핵심 원리

GKD에는 서로 독립적인 두 축이 있다. `lambda`는 고정 state와 student state를 섞는 비율이고, `beta`는 그 state에서 generalized JSD의 모양을 정한다. 이 강좌의 convention에서 `beta=0`은 forward KL, `beta=1`은 reverse KL이다.

원 GKD는 선택한 prefix에서 **vocabulary 전체 분포**를 비교한다. 뒤의 현대 sampled OPD는 student가 뽑은 token만으로 score-function estimator를 만들 수 있다. 둘 다 on-policy state를 쓸 수 있지만 estimator와 분산이 같지 않다.

### 실제 구현: 왜 이렇게 만들었나

collection에서 lambda로 state source를 고르고 loss에서 beta로 divergence를 고른다. 두 선택을 함수 하나에 숨기지 않아 ablation이 가능하다. teacher는 `eval()`과 `no_grad()`로 score하고 student만 update한다.

실제 코드: [`core.py`](../../src/opd_study/training/core.py), [`losses.py`](../../src/opd_study/algorithms/losses.py).

In [2]:
import inspect
from opd_study.algorithms import generalized_kd_loss

objects_to_show = (generalized_kd_loss,)
for object_to_show in objects_to_show:
    source_lines = inspect.getsource(object_to_show).splitlines()
    print(f"\n# {object_to_show.__module__}.{object_to_show.__qualname__}")
    print("\n".join(source_lines[:80]))
    if len(source_lines) > 80:
        print(f"... {len(source_lines) - 80} more lines; open the linked source file")


# opd_study.algorithms.losses.generalized_kd_loss
def generalized_kd_loss(
    student_logits: Tensor,
    trajectories: TrajectoryBatch,
    signals: TeacherSignals,
    *,
    beta: float = 0.5,
    temperature: float = 1.0,
) -> LossOutput:
    """Generalized JSD objective from GKD on whatever states were collected.

    State-source mixing (the GKD lambda) belongs in collection, not in this loss.
    Keeping those mechanisms separate makes on-policy/off-policy ablations auditable.
    """

    shifted_student, _, _, prediction_mask = shifted_causal_tensors(
        student_logits,
        trajectories.token_ids,
        trajectories.attention_mask,
        trajectories.response_mask,
    )
    shifted_teacher = _teacher_shifted_logits(signals, trajectories, student_logits)
    shifted_loss = torch.mul(
        generalized_jsd_from_logits(
            shifted_teacher,
            shifted_student,
            beta=beta,
            temperature=temperature,
        ),
        tempera

### 다른 선택지는 없나?

lambda는 고정값, warm-up schedule, 성능 기반 adaptive schedule이 가능하다. beta도 FKL/RKL/JSD 사이를 고를 수 있다. 기본은 논문 의미를 먼저 재현하는 고정값이며 adaptive 정책은 별도 실험 변수로 둔다.

### 2/3 · 실행하고 관찰하기

실행 전 예측: L04의 첫 출력에서 가장 먼저 확인해야 할 invariant는 무엇일까? 한 문장으로 적고 실행한다.

In [3]:
from opd_study.math import generalized_jsd_from_logits

teacher = torch.randn(2, 5, 7)
student = torch.randn(2, 5, 7, requires_grad=True)
for beta in (0.0, 0.5, 1.0):
    value = generalized_jsd_from_logits(teacher, student, beta=beta).mean()
    print(f"beta={beta}: {value.item():.5f}")
print("beta=0 is forward KL; beta=1 is reverse KL in this repository's convention.")

beta=0.0: 0.60619
beta=0.5: 0.13255
beta=1.0: 0.60986
beta=0 is forward KL; beta=1 is reverse KL in this repository's convention.


In [4]:
generator = torch.Generator().manual_seed(42)
lambda_on_policy = 0.5
state_sources = ["student" if torch.rand((), generator=generator) < lambda_on_policy
                 else "fixed" for _ in range(8)]
print("GKD state sources:", state_sources)
print("lambda chooses states; beta chooses the divergence. They are different knobs.")

GKD state sources: ['fixed', 'fixed', 'student', 'fixed', 'student', 'fixed', 'student', 'fixed']
lambda chooses states; beta chooses the divergence. They are different knobs.


## Checks

In [5]:
forward = generalized_jsd_from_logits(teacher, student, beta=0.0)
reverse = generalized_jsd_from_logits(teacher, student, beta=1.0)
assert forward.shape == reverse.shape == (2, 5)
assert set(state_sources) == {"fixed", "student"}
print("check passed: [B,T,V] -> [B,T], with separate lambda and beta")

check passed: [B,T,V] -> [B,T], with separate lambda and beta


**연습 (8분):** seed를 고정한 채 lambda를 0, 0.5, 1로 바꾸고 state-source trace를 비교하라. beta는 그대로 두고 두 knob가 독립임을 기록하라.

<details><summary>확인 기준</summary>lambda 0은 fixed, 1은 student state만 고르며 beta/JSD 수식은 변하지 않는다.</details>

## 내가 자주 틀리는 것

### M1 — lambda와 beta를 같은 knob로 설명하기

- 틀린 형태: lambda를 올리면 reverse KL이 된다고 말한다.
- 왜 틀렸나: lambda는 state mixture, beta는 divergence다.
- 고친 형태: collection과 loss config를 별도 열로 기록한다.
- 관련 검사: `test_gjsd_boundaries_have_named_kl_direction`

### M2 — GKD와 sampled policy-gradient OPD를 동일시하기

- 틀린 형태: on-policy prefix라는 이유만으로 estimator도 같다고 한다.
- 왜 틀렸나: full vocabulary와 sampled token estimator는 분산·메모리가 다르다.
- 고친 형태: state source와 estimator를 두 축으로 분류한다.
- 관련 검사: `test_rollout_snapshots_are_detached_and_mode_is_restored`

## 60초 요약

1. GKD의 lambda와 beta를 분리한다
2. 일반화 JSD를 구현한다
3. gradient 경로를 추적한다

## Next Steps

다음 노트북으로 가기 전, 위 assertion을 다시 실행하고 틀린 예측 한 줄을 남긴다.

### Sources

- [`gkd`](https://arxiv.org/abs/2306.13649v3) · `2306.13649v3` · license `CC-BY-4.0` · [audited manifest](../../docs/sources.yml)
- [`trl_gkd_reference`](https://github.com/huggingface/trl) · `1e3ba4e80dfd8c64f11022a7ae47de6a58255ca5` · license `Apache-2.0` · [audited manifest](../../docs/sources.yml)